# Multitask training — GS40 + GS55 combined

Train **one** ConvNeXt + StarDist multitask model on tiles from both gestational ages.

## Prerequisites

For **each** cohort (GS40 and GS55) you must have already run the preprocessing pipeline from `train_multitask_GS40_paths.ipynb` (or the GS55 equivalent):

- PNG tiles under `…/train/images` (and the same folder for val if you use the GS40 layout)
- Instance masks + `*_inst2class.json` sidecars under `…/stardist_multitask_ready/train_instance_labels`
- Split CSVs (`train.csv` / `val.csv`) with one tile stem per row

## Quick start

1. Edit **section 2 — PARAMETERS** (paths for GS40 + GS55, hyperparameters).
2. Run sections **3→4** (load stems, write `config_gs40_gs55_multitask.yaml`).
3. **Run training:** use **section 5** — terminal (recommended) or the notebook cell below.
4. **Fine-tune only:** set `RESUME_CHECKPOINT` in PARAMETERS, then re-run **2→5** (parameters through run).

## How this notebook works

1. **Parameters** — set paths for both cohorts and hyperparameters.
2. **Load stems** — reads train/val stems from each cohort’s split CSVs and **concatenates** them.
3. **Write config** — builds a YAML where `data.train_sources` / `data.val_sources` list two roots. `train_v2.py` concatenates datasets internally and **filters** combined `train_stems` / `val_stems` per `images_dir` (GS40 stems are not required to exist under GS55, and vice versa).
4. **Run** — launches `python -m shared_convnext_stardist_decoder.train_v2`.

## Single cohort only

Point both sources at the same paths, or use `train_multitask_GS40_paths.ipynb` instead.

---
## 1 · Imports

In [2]:
from pathlib import Path

from train_utils import build_training_config, read_stems, run_training, write_config

print("OK — train_utils loaded")

OK — train_utils loaded


---
## 2 · PARAMETERS  ← edit here

**Image / label dirs** must match what you used when building `inst2class` files (same stems must exist in both `images` and `train_instance_labels`).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GS40 (gestational 40)
# ═══════════════════════════════════════════════════════════════════════════
GS40_ROOT = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\dataset_256_40k_48_slides")
GS40_TRAIN_IMAGES = GS40_ROOT / "train" / "images"
GS40_INST_LABELS  = GS40_ROOT / "stardist_multitask_ready" / "train_instance_labels"
# GS40 val tiles live in the same image folder as train; only stems differ
GS40_VAL_IMAGES   = GS40_TRAIN_IMAGES
GS40_VAL_LABELS   = GS40_INST_LABELS
GS40_SPLIT_TRAIN  = GS40_ROOT / "splits" / "fold_0" / "train.csv"
GS40_SPLIT_VAL    = GS40_ROOT / "splits" / "fold_0" / "val.csv"

# ═══════════════════════════════════════════════════════════════════════════
# GS55 — labels live in train/labels and val/labels (different from GS40 layout)
# ═══════════════════════════════════════════════════════════════════════════
GS55_ROOT = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\dataset_256_40k_GS55")
GS55_TRAIN_IMAGES = GS55_ROOT / "train" / "images"
GS55_INST_LABELS  = GS55_ROOT / "train" / "labels"   # fixed: was stardist_multitask_ready/train_instance_labels
GS55_VAL_IMAGES   = GS55_ROOT / "val" / "images"
GS55_VAL_LABELS   = GS55_ROOT / "val" / "labels"     # fixed: was same wrong path as train labels
GS55_SPLIT_TRAIN  = GS55_ROOT / "splits" / "fold_0" / "train.csv"
GS55_SPLIT_VAL    = GS55_ROOT / "splits" / "fold_0" / "val.csv"

# ═══════════════════════════════════════════════════════════════════════════
# Output & repo
# ═══════════════════════════════════════════════════════════════════════════
REPO_ROOT   = Path(r"C:\Users\Andre\cursor_projects\Convnext_stardist")
CONFIG_PATH = REPO_ROOT / "shared_convnext_stardist_decoder" / "config_gs40_gs55_multitask.yaml"
CKPT_OUT    = GS40_ROOT / "convnext_stardist_multitask_runs" / "run_gs40_gs55"

EXPERIMENT_NAME = "convnext_stardist_mt_gs40_gs55_v1"

# Class names — same order as GS40 training (lowercase "gi" for model compatibility)
CLASS_NAMES = [
    "bone", "brain", "eye", "heart", "lungs", "gi", "liver", "spleen",
    "pancreas", "kidney", "mesokidney", "collagen", "ear", "nontissue",
    "thymus", "thyroid", "bladder", "skull", "spleen2",
]

# Training hyperparameters (tweak as needed)
EPOCHS                 = 35
BATCH_SIZE             = 8
LR                     = 5e-5
LR_MIN                 = 1e-6
WEIGHT_DECAY           = 0.03
FREEZE_BACKBONE_EPOCHS = 10
LOSS_W_PROB            = 2.0
LOSS_W_DIST            = 0.15
LOSS_W_CLS             = 1.0
LOSS_W_INST            = 0.25
CLS_SEMANTIC_DIM       = 128  # widened from 64: more stage-4 texture signal to classifier
NUM_WORKERS            = 8

# Fine-tuning: set to a .pt path to resume, else None
RESUME_CHECKPOINT = None
RESUME_STRICT     = True

CKPT_OUT.mkdir(parents=True, exist_ok=True)
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Config → {CONFIG_PATH}")
print(f"Checkpoints → {CKPT_OUT}")
print(f"GS55 train labels → {GS55_INST_LABELS}")
print(f"GS55 val labels   → {GS55_VAL_LABELS}")

---
## 3 · Load and merge train / val stems

In [4]:
def _load_split(csv_path: Path, label: str) -> list[str]:
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing {label}: {csv_path}")
    stems = read_stems(csv_path)
    print(f"{label}: {len(stems):,} stems from {csv_path.name}")
    return stems

train_40 = _load_split(GS40_SPLIT_TRAIN, "GS40 train")
val_40   = _load_split(GS40_SPLIT_VAL,   "GS40 val")
train_55 = _load_split(GS55_SPLIT_TRAIN, "GS55 train")
val_55   = _load_split(GS55_SPLIT_VAL,   "GS55 val")

train_stems = train_40 + train_55
val_stems   = val_40 + val_55

# De-duplicate while preserving order (should rarely matter)
def _dedupe(xs: list[str]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for s in xs:
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out

train_stems = _dedupe(train_stems)
val_stems   = _dedupe(val_stems)

overlap = set(train_stems) & set(val_stems)
print(f"\nCombined: train {len(train_stems):,}  |  val {len(val_stems):,}")
print(f"Train∩Val stem overlap: {len(overlap)} tiles (should be 0)")

GS40 train: 38,716 stems from train.csv
GS40 val: 7,992 stems from val.csv
GS55 train: 16,384 stems from train.csv
GS55 val: 13,566 stems from val.csv

Combined: train 55,100  |  val 21,558
Train∩Val stem overlap: 0 tiles (should be 0)


---
## 4 · Build config and write YAML

`build_training_config` is used for model / train / checkpoint defaults. The **`data`** section is then replaced with `train_sources` + `val_sources` so both cohorts are loaded.

In [5]:
cfg = build_training_config(
    experiment_name=EXPERIMENT_NAME,
    train_images_dir=GS40_TRAIN_IMAGES,
    inst_labels_dir=GS40_INST_LABELS,
    ckpt_out=CKPT_OUT,
    train_stems=train_stems,
    val_stems=val_stems,
    class_names=CLASS_NAMES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    lr_min=LR_MIN,
    weight_decay=WEIGHT_DECAY,
    freeze_backbone_epochs=FREEZE_BACKBONE_EPOCHS,
    loss_w_cls=LOSS_W_CLS,
    loss_w_inst=LOSS_W_INST,
    loss_w_dist=LOSS_W_DIST,
    cls_semantic_dim=CLS_SEMANTIC_DIM,
)

cfg["data"] = {
    "train_stems": train_stems,
    "val_stems": val_stems,
    "train_sources": [
        {"images_dir": str(GS40_TRAIN_IMAGES), "labels_dir": str(GS40_INST_LABELS)},
        {"images_dir": str(GS55_TRAIN_IMAGES), "labels_dir": str(GS55_INST_LABELS)},
    ],
    "val_sources": [
        {"images_dir": str(GS40_VAL_IMAGES), "labels_dir": str(GS40_VAL_LABELS)},
        {"images_dir": str(GS55_VAL_IMAGES), "labels_dir": str(GS55_VAL_LABELS)},
    ],
    "cache_to_ram": False,
}

cfg["train"]["loss_w_prob"] = LOSS_W_PROB
cfg["train"]["num_workers"] = NUM_WORKERS

write_config(cfg, CONFIG_PATH)
print("\nMulti-source data section:")
print("  train_sources:", len(cfg["data"]["train_sources"]), "roots")
print("  val_sources:  ", len(cfg["data"]["val_sources"]), "roots")

Config written: C:\Users\Andre\cursor_projects\Convnext_stardist\shared_convnext_stardist_decoder\config_gs40_gs55_multitask.yaml
  experiment : convnext_stardist_mt_gs40_gs55_v1
  stems      : train 55100 / val 21558
  classes    : ['bone', 'brain', 'eye', 'heart', 'lungs', 'gi', 'liver', 'spleen', 'pancreas', 'kidney', 'mesokidney', 'collagen', 'ear', 'nontissue', 'thymus', 'thyroid', 'bladder', 'skull', 'spleen2']
  epochs     : 35  batch: 8  lr: 5e-05 → 1e-06
  freeze_bb  : 10 epochs
  loss w_cls/w_inst: 1.0/0.25
  ckpt dir   : \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\dataset_256_40k_48_slides\convnext_stardist_multitask_runs\run_gs40_gs55

Multi-source data section:
  train_sources: 2 roots
  val_sources:   2 roots


---
## 4b · Write finetune config (same splits, new experiment)

Writes `config_finetune_gs40_gs55.yaml` using **identical** `train_stems` / `val_stems` from section 3 so the fine-tune sees exactly the same 55,100 / 21,558 tile split as the original run.

In [ ]:
FINETUNE_CONFIG_PATH = REPO_ROOT / "shared_convnext_stardist_decoder" / "config_finetune_gs40_gs55.yaml"
FINETUNE_CKPT_OUT    = GS40_ROOT / "convnext_stardist_multitask_runs" / "run_gs40_gs55_finetune_e14"
FINETUNE_EXPERIMENT  = "convnext_stardist_mt_gs40_gs55_finetune_e14"

ft_cfg = build_training_config(
    experiment_name=FINETUNE_EXPERIMENT,
    train_images_dir=GS40_TRAIN_IMAGES,
    inst_labels_dir=GS40_INST_LABELS,
    ckpt_out=FINETUNE_CKPT_OUT,
    train_stems=train_stems,
    val_stems=val_stems,
    class_names=CLASS_NAMES,
    epochs=15,
    batch_size=BATCH_SIZE,
    lr=1e-5,
    lr_min=1e-7,
    weight_decay=WEIGHT_DECAY,
    freeze_backbone_epochs=5,
    loss_w_cls=LOSS_W_CLS,
    loss_w_inst=LOSS_W_INST,
    loss_w_dist=LOSS_W_DIST,
    cls_semantic_dim=CLS_SEMANTIC_DIM,
)

ft_cfg["data"] = {
    "train_stems": train_stems,
    "val_stems":   val_stems,
    "train_sources": [
        {"images_dir": str(GS40_TRAIN_IMAGES), "labels_dir": str(GS40_INST_LABELS)},
        {"images_dir": str(GS55_TRAIN_IMAGES), "labels_dir": str(GS55_INST_LABELS)},
    ],
    "val_sources": [
        {"images_dir": str(GS40_VAL_IMAGES),   "labels_dir": str(GS40_VAL_LABELS)},
        {"images_dir": str(GS55_VAL_IMAGES),   "labels_dir": str(GS55_VAL_LABELS)},
    ],
    "cache_to_ram": False,
}

ft_cfg["train"]["loss_w_prob"] = LOSS_W_PROB
ft_cfg["train"]["num_workers"] = NUM_WORKERS

write_config(ft_cfg, FINETUNE_CONFIG_PATH)
print("\nFinetune config written with same splits as original run.")

---
## 5 · Run training

**From terminal (recommended — live progress visible):**

```powershell
conda activate convnext_stardist_mt
cd C:\Users\Andre\cursor_projects\Convnext_stardist

# Fresh run (GS40 + GS55 — uses config written in section 4)
python -m shared_convnext_stardist_decoder.train_v2 --config shared_convnext_stardist_decoder\config_gs40_gs55_multitask.yaml

# Fine-tune from best checkpoint of v1 run (epoch 14, val_loss=0.44122)
python -m shared_convnext_stardist_decoder.train_v2 ^
    --config shared_convnext_stardist_decoder\config_finetune_gs40_gs55.yaml ^
    --resume "\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\dataset_256_40k_48_slides\convnext_stardist_multitask_runs\run_gs40_gs55\convnext_stardist_mt_gs40_gs55_v1\best.pt"

# Warm-start from V1 (partial load: seg decoder only, cls head re-initialised)
python -m shared_convnext_stardist_decoder.train_v2 `
    --config shared_convnext_stardist_decoder\config_gs40_gs55_multitask.yaml `
    --resume <path\to\v1_best.pt> --resume_strict false
```

**What to watch in the log:**

| Column | Good sign |
|--------|-----------|
| `bce` | Decreases epoch 1→50 |
| `cls_px` | Decreases from ~1.5 toward <0.8 |
| `val cls_px` | Tracks train; large gap = overfitting → use `best.pt` |
| `dist` | Decreases toward <1.0 |

> **Fine-tuning tip:** set `RESUME_CHECKPOINT` in **section 2 (PARAMETERS)**, then re-run **sections 2→5** (parameters through this run cell), or refresh the YAML only (sections **2→4**) if you launch training from the terminal.

In [ ]:
# Run inside notebook — or use the terminal command in section 5 instead.
# Set RESUME_CHECKPOINT and RESUME_STRICT in PARAMETERS (section 2) to fine-tune.

# exit_code = run_training(
#     config_path       = CONFIG_PATH,
#     repo_root         = REPO_ROOT,
#     resume_checkpoint = RESUME_CHECKPOINT,
#     resume_strict     = RESUME_STRICT,
# )
# assert exit_code == 0, f"Training failed with exit code {exit_code}"